# Chạy thử nhanh PS-IPPO trên Google Colab

Notebook này chạy **toàn bộ quy trình ở quy mô nhỏ** (tiền xử lý → kiểm thử → tune baseline → train vài trăm episode → đánh giá trên test) để xem nhanh kết quả trước khi chạy đợt cuối bằng `Khoa_luan_Colab.ipynb`.

| Nơi | Vai trò |
|---|---|
| GitHub `MinhLysA/Khoa_luan` | mã nguồn + `config.yaml`, clone vào `/content/Khoa_luan` (không lưu lên Drive) |
| `MyDrive/KLTN_data/` | 3 file CSV gốc M5 (giống notebook chính) |
| `MyDrive/KLTN_test/` | **kết quả lần chạy mới nhất**, mỗi lần chạy **xóa bản cũ và ghi đè** |

`KLTN_test` tách riêng khỏi `KLTN_final` nên chạy thử không ảnh hưởng kết quả đợt cuối.

**Thời gian (GPU T4):** khoảng 10–15 phút với mặc định 300 episode.

**Cách chạy:** Runtime → Change runtime type → **T4 GPU**, rồi Runtime → **Run all**. Kết quả chỉ được chép lên Drive ở **Bước 9** (cuối cùng), nên bản cũ trên Drive được giữ nguyên nếu lần chạy mới bị lỗi giữa chừng.

> Kết quả chạy thử **chỉ để kiểm tra quy trình**, không dùng cho báo cáo: 300 episode chưa hội tụ (mô hình chính cần khoảng 3.000–4.000), và chỉ đánh giá 5 lần.

## Bước 1. Kết nối Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Bước 2. Tham số chạy thử và lấy mã nguồn từ GitHub

Sửa tham số ở đầu cell nếu cần. Mỗi lần chạy clone mới mã nguồn, mọi đầu ra ghi vào `/content` trước, đến Bước 9 mới chép lên Drive.

In [ ]:
import os, shutil
EPISODES      = 300    # so episode train (mo hinh chinh dung 4.000)
EVAL_EPISODES = 5      # so lan danh gia tren mien test (dot cuoi dung 30)
TUNE_EPISODES = 1      # so episode moi cau hinh khi tune baseline (dot cuoi dung 3)
TAG           = "quick"

REPO_URL = "https://github.com/MinhLysA/Khoa_luan.git"   # repo private: https://<TOKEN>@github.com/MinhLysA/Khoa_luan.git
CODE_ROOT = "/content/Khoa_luan"
CODE_DIR  = f"{CODE_ROOT}/rl_inventory"
DATA_DIR  = "/content/drive/MyDrive/KLTN_data"
OUT_DIR   = "/content/drive/MyDrive/KLTN_test"

os.chdir("/content")
shutil.rmtree(CODE_ROOT, ignore_errors=True)
!git clone -q --depth 1 {REPO_URL} {CODE_ROOT}
!git -C {CODE_ROOT} log -1 --format="Ma nguon lay tu GitHub: %h (%ad) %s" --date=short
os.chdir(CODE_DIR)
for d in ["checkpoints", "results", "runs"]:        # bo ket qua cu co san trong repo
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d)
os.makedirs("logs", exist_ok=True)

import yaml
print("include_price_signal =", yaml.safe_load(open("config.yaml", encoding="utf-8"))["env"].get("include_price_signal"))
print("Thu muc lam viec:", os.getcwd())

## Bước 3. Cài thư viện và kiểm tra GPU

In [ ]:
import os; os.chdir("/content/Khoa_luan/rl_inventory")  # neu loi (Colab vua khoi dong lai): chay lai Buoc 1-3
!pip install -q -r requirements.txt
import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| PyTorch", torch.__version__)
else:
    print("!!! KHONG CO GPU -> van chay duoc tren CPU nhung cham hon khoang 3-4 lan.")

## Bước 4. Lấy dữ liệu gốc M5 từ Drive và tiền xử lý

Giống Bước 4 của notebook chính: đọc 3 file CSV trong `MyDrive/KLTN_data/`, tiền xử lý theo `config.yaml`, rồi đối chiếu với bản đã xử lý trên GitHub.

In [ ]:
import os; os.chdir("/content/Khoa_luan/rl_inventory")  # neu loi (Colab vua khoi dong lai): chay lai Buoc 1-3
import shutil, numpy as np, yaml
CAN = ["sales_train_evaluation.csv", "calendar.csv", "sell_prices.csv"]
thieu = [f for f in CAN if not os.path.exists(f"{DATA_DIR}/{f}")]
if thieu:
    raise SystemExit(f"Thieu {thieu} trong {DATA_DIR}. Tai tu Kaggle (M5 Forecasting - Accuracy) roi upload vao day.")

GIT_COPY = "/content/processed_tu_github"
shutil.rmtree(GIT_COPY, ignore_errors=True)
shutil.copytree("data/processed", GIT_COPY)

p = yaml.safe_load(open("config.yaml", encoding="utf-8"))
THAM_SO = (f"--n_warehouses {p['env']['n_warehouses']} --n_skus {p['env']['n_skus']} "
           f"--min_mean_demand {p['preprocess']['min_mean_demand']} --split_day {p['env']['split_day']}")
!python -u scripts/data_preprocessing.py --m5 --raw_dir "{DATA_DIR}" --output_dir data/processed {THAM_SO} 2>&1 | tee logs/tien_xu_ly.log | tail -3

for f in ["demand_data", "calendar_features", "price_per_pair", "price_series"]:
    a, b = np.load(f"{GIT_COPY}/{f}.npy"), np.load(f"data/processed/{f}.npy")
    same = a.shape == b.shape and np.allclose(a, b, equal_nan=True)
    print(f"  {f:<18} {'TRUNG KHOP' if same else '!!! KHAC - kiem tra lai file goc tren Drive'}")

## Bước 5. Kiểm thử mã nguồn

Phải thấy **`44 passed`**. Nếu có `failed` thì dừng lại.

In [ ]:
import os; os.chdir("/content/Khoa_luan/rl_inventory")  # neu loi (Colab vua khoi dong lai): chay lai Buoc 1-3
r = !python -m pytest tests -q 2>&1
print(r[-1])
assert "failed" not in r[-1] and "passed" in r[-1], "Test FAIL - xem chi tiet: !python -m pytest tests -q"


## Bước 6. Tune 3 baseline trên miền validation (bộ chung và theo nhóm quy mô cầu)

In [ ]:
import os; os.chdir("/content/Khoa_luan/rl_inventory")  # neu loi (Colab vua khoi dong lai): chay lai Buoc 1-3
LOC = "=>|theo nhom|: [{]"   # tach ra bien: dau { ngay trong dong ! lam IPython bo qua viec thay {TUNE_EPISODES}
!python -u scripts/tune_baselines.py --episodes {TUNE_EPISODES} --tune_mode val --per_group 2>&1 | tee logs/tune_baselines.log | grep -E "{LOC}"
assert os.path.exists("results/baseline_params.json"), "Tune baseline LOI - xem logs/tune_baselines.log"


## Bước 7. Huấn luyện PS-IPPO

Đầu log phải có `device=cuda  obs_per_pair=44`.

In [ ]:
import os; os.chdir("/content/Khoa_luan/rl_inventory")  # neu loi (Colab vua khoi dong lai): chay lai Buoc 1-3
import time
t = time.time()
!python -u scripts/train.py --episodes {EPISODES} --tag {TAG} 2>&1 | tee logs/train.log | grep -E "device=|Ep +[0-9]*[05]0/|checkpoint :|KHONG checkpoint|Error"
print(f"=> {(time.time() - t) / 60:.1f} phut, {(time.time() - t) / EPISODES:.2f} giay/episode")
assert os.path.exists(f"checkpoints/best_model_{TAG}.pth"), "Train LOI - xem logs/train.log"

## Bước 8. Đánh giá trên miền test và xem kết quả

In [ ]:
import os; os.chdir("/content/Khoa_luan/rl_inventory")  # neu loi (Colab vua khoi dong lai): chay lai Buoc 1-3
!python -u scripts/evaluate.py --checkpoint checkpoints/best_model_{TAG}.pth --episodes {EVAL_EPISODES} --tag {TAG} 2>&1 | tee logs/evaluate.log | grep -A12 "^Policy"
!python scripts/plot_learning_curve.py --tag {TAG}

from IPython.display import Image, display
import glob
for png in sorted(glob.glob(f"results/*{TAG}*.png")):
    print(png); display(Image(png))

## Bước 9. Lưu kết quả lên Drive (ghi đè lần chạy trước)

Xóa nội dung cũ trong `MyDrive/KLTN_test` (giữ thư mục nên link chia sẻ không đổi) rồi chép toàn bộ kết quả lần này: `results/`, `checkpoints/`, `runs/` (TensorBoard), `logs/` và `thong_tin_lan_chay.txt` (commit, thời điểm, tham số).

In [ ]:
import os; os.chdir("/content/Khoa_luan/rl_inventory")  # neu loi (Colab vua khoi dong lai): chay lai Buoc 1-3
import datetime, subprocess, json
commit = subprocess.run(["git", "-C", CODE_ROOT, "log", "-1", "--format=%h %ad %s", "--date=short"],
                        capture_output=True, text=True).stdout.strip()
with open("logs/thong_tin_lan_chay.txt", "w", encoding="utf-8") as f:
    f.write(f"Thoi diem : {datetime.datetime.now():%Y-%m-%d %H:%M}\n"
            f"Commit    : {commit}\n"
            f"Episode   : {EPISODES} | danh gia: {EVAL_EPISODES} | tune: {TUNE_EPISODES}\n"
            f"GPU       : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}\n")

# Xoa NOI DUNG lan chay cu nhung GIU thu muc KLTN_test -> link chia se tren Drive khong doi
os.makedirs(OUT_DIR, exist_ok=True)
for old in os.listdir(OUT_DIR):
    p = f"{OUT_DIR}/{old}"
    shutil.rmtree(p) if os.path.isdir(p) else os.remove(p)
for d in ["results", "checkpoints", "runs", "logs"]:
    shutil.copytree(d, f"{OUT_DIR}/{d}")
print(open("logs/thong_tin_lan_chay.txt", encoding="utf-8").read())
print("Da luu vao", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))